[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yy726/GPQA-eval/blob/main/notebooks/02_sft_dolci_lora.ipynb)

# 02 · 基线：用 Dolci 做 LoRA SFT / Baseline: LoRA SFT on Dolci

流程：Dolci-Think-SFT 流式读取 → 按来源和长度过滤、抽样 → 13-gram 去污染 → LoRA SFT（fp16）→ 合并 → 推到 HF Hub → 用 `01` 评测。

这是**起点**，不是最优解。可以尝试的方向：
- 换数据来源和比例：科学 vs 数学 vs 代码，用 `--list-sources` 看有哪些来源
- 要不要保留 `<think>` 思维链？评测只给 4096 token 的生成预算，思维链太长会被截断
- 样本数、长度上限、学习率、LoRA rank
- 先 SFT 再接 `03` 的 GRPO

**时间预算：训练总墙钟时间 ≤ 4 小时（规则）。** Colab 可能断线，所以 checkpoint 会自动推到 Hub，断了可以续训。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# 克隆比赛仓库（脚本都在里面） / clone the hackathon repo
import os
if not os.path.exists("GPQA-eval"):
    !git clone -q https://github.com/yy726/GPQA-eval.git
%cd GPQA-eval

In [ ]:
# 训练环境（Unsloth + TRL + PEFT）。不要和评测在同一个 runtime 里装：vLLM 与 Unsloth 依赖的 torch 版本可能不同。
!pip install -q -r requirements-train.txt
import importlib.metadata as md
for p in ["unsloth", "trl", "peft", "transformers", "torch"]:
    print(p, md.version(p))

In [ ]:
# GPQA 是 gated 数据集：先在 https://huggingface.co/datasets/Idavidrein/gpqa 接受条款，
# 再用有 read（提交模型需要 write）权限的 token 登录。
# 推荐：Colab 左侧 🔑 Secrets 里添加 HF_TOKEN，这里就会自动读取。
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))

In [ ]:
# ==== 配置 / config ====
SMOKE        = False                       # True = 只跑 50 步（助教冒烟测试用）
INIT_MODEL   = "Qwen/Qwen3-4B-Base"      # 或 "Qwen/Qwen3-1.7B-Base"（快 2 倍以上，适合迭代）
DATASET      = "allenai/Dolci-Think-SFT-7B"   # 也可以试 "allenai/Dolci-Instruct-SFT-7B"
INCLUDE      = ["math", "science", "stem"]    # 来源名里包含这些关键词才保留；[] 表示不过滤
EXCLUDE      = ["safety", "wildguard"]
N_SAMPLES    = 300 if SMOKE else 5000
MAX_SEQ_LEN  = 4096
LORA_R       = 16
LR           = 1e-4
EPOCHS       = 1
USE_UNSLOTH  = True
HUB_REPO     = "my-team/qwen3-4b-dolci-sft"   # 你的 HF 用户名/仓库名（会公开）
EXTRA_LOAD_KWARGS = {}
PUSH         = not SMOKE

## 1. 看看 Dolci 里有哪些来源（可选）

In [ ]:
!python scripts/prepare_dolci_sft.py --dataset "$DATASET" --out /tmp/x.jsonl --list-sources 20000

## 2. 构建训练集

In [ ]:
inc = " ".join(INCLUDE); exc = " ".join(EXCLUDE)
!python scripts/prepare_dolci_sft.py --dataset "$DATASET" --out data/sft.jsonl --n $N_SAMPLES \
    --tokenizer "$INIT_MODEL" --max-tokens $MAX_SEQ_LEN --include $inc --exclude $exc

## 3. 去污染（必须，提交时要填结果）

In [ ]:
!python scripts/decontam.py --train-jsonl data/sft.jsonl --out data/decontam.json
import json
rep = json.load(open("data/decontam.json"))
bad = set(rep["flagged_row_indices"])
rows = [l for i, l in enumerate(open("data/sft.jsonl")) if i not in bad]
open("data/sft_clean.jsonl", "w").writelines(rows)
print(f"kept {len(rows)} rows, dropped {len(bad)}; overlap_rate={rep['overlap_rate']}")

## 4. 加载模型 + LoRA

In [ ]:
# 加载模型 + LoRA。默认走 Unsloth（更快、更省显存）；如果 Unsloth 在你的环境里报错，把 USE_UNSLOTH 改成 False，
# 会回退到 transformers + PEFT（慢一些，但依赖更少）。T4 没有 bf16，一律 fp16。
import torch

TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    model, tok = FastLanguageModel.from_pretrained(
        model_name=INIT_MODEL,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,          # 4B 的 fp16 权重约 8GB，T4 放得下，不需要 QLoRA
        dtype=None,                  # 自动选择：T4 上是 float16
        **EXTRA_LOAD_KWARGS,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=2 * LORA_R,
        lora_dropout=0,
        target_modules=TARGET_MODULES,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=0,
    )
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model
    tok = AutoTokenizer.from_pretrained(INIT_MODEL)
    model = AutoModelForCausalLM.from_pretrained(INIT_MODEL, dtype=torch.float16, device_map={"": 0})
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.0, task_type="CAUSAL_LM", target_modules=TARGET_MODULES,
    ))

if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model.print_trainable_parameters()
print(f"GPU mem after load: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")

## 5. 训练

In [ ]:
import time, json, torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

ds = load_dataset("json", data_files="data/sft_clean.jsonl", split="train")
n_train_tokens = sum(ds["n_tokens"])
ds = ds.remove_columns([c for c in ds.column_names if c not in ("prompt", "completion")])

cfg = SFTConfig(
    output_dir="outputs/sft",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    max_steps=50 if SMOKE else -1,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    max_length=MAX_SEQ_LEN,
    fp16=True, bf16=False,              # T4: fp16 only
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    push_to_hub=PUSH,                   # checkpoint 自动推到 Hub，断线可续训
    hub_model_id=HUB_REPO + "-ckpt",
    hub_strategy="checkpoint",
    hub_private_repo=True,
    report_to="none",
    seed=0,
)
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds, processing_class=tok)

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
# 断线后重新运行：把 resume_from_checkpoint 设为 True（或从 Hub 的 -ckpt 仓库下载 last-checkpoint）
out = trainer.train(resume_from_checkpoint=False)
dt = time.time() - t0
seen = n_train_tokens * EPOCHS if not SMOKE else None
print(f"train time: {dt / 3600:.2f} h   peak mem: {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")
if seen:
    print(f"throughput ≈ {seen / dt:.0f} tok/s  (记到提交 JSON 的 train_gpu_hours_t4: {dt / 3600:.2f})")
else:
    print(f"steps/s: {out.metrics.get('train_steps_per_second')}")

## 6. 快速自检（用非 GPQA 的题目）

In [ ]:
msgs = [{"role": "user", "content": "What is the correct answer to this question: Which particle mediates the electromagnetic force?\nChoices:\n(A) gluon\n(B) photon\n(C) W boson\n(D) graviton\nLet's think step by step: "}]
inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
with torch.no_grad():
    gen = model.generate(**inputs, max_new_tokens=512, do_sample=False)
print(tok.decode(gen[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## 7. 合并并推到 Hub

In [ ]:
# 合并 LoRA 并以 fp16 推到 Hub（公开仓库，排行榜要求）
if PUSH:
    if USE_UNSLOTH:
        model.push_to_hub_merged(HUB_REPO, tok, save_method="merged_16bit", private=False)
    else:
        merged = model.merge_and_unload()
        merged.push_to_hub(HUB_REPO, private=False)
        tok.push_to_hub(HUB_REPO, private=False)
    print(f"pushed: https://huggingface.co/{HUB_REPO}")
    print("下一步：在一个新的 runtime 里打开 01_eval_gpqa.ipynb，MODEL 填上面的仓库名。")